In [ ]:
# Run this cell first — defines client, model, and chat (single place avoids NameError).
from pathlib import Path
import os

from dotenv import load_dotenv
from anthropic import Anthropic


def _project_root() -> Path:
    """Find folder containing this project (notebook or dataset), even if cwd differs."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents[:15]]:
        if (d / "code_grader.ipynb").is_file():
            return d
    for d in [here, *here.parents[:15]]:
        if (d / "dataset.json").is_file():
            return d
    return here


_PROJECT = _project_root()
load_dotenv(_PROJECT / ".env")
load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Put it in a .env file in this folder, or export it before "
        "starting Jupyter / Cursor, then restart the kernel."
    )

client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def _text_from_message(message) -> str:
    parts: list[str] = []
    for block in message.content:
        if getattr(block, "type", None) == "text" and getattr(block, "text", None):
            parts.append(block.text)
    return "".join(parts)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    if stop_sequences is None:
        stop_sequences = []
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if system:
        params["system"] = system

    msg = client.messages.create(**params)
    return _text_from_message(msg)

In [ ]:
import json
from pathlib import Path


def _parse_json_object(raw: str) -> dict:
    """Parse first JSON object from model text (handles fences and leading prose)."""
    s = raw.strip()
    for prefix in ("```json", "```JSON", "```"):
        if s.startswith(prefix):
            s = s[len(prefix) :].strip()
    if s.endswith("```"):
        s = s[:-3].strip()

    dec = json.JSONDecoder()
    start = s.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in grader output: {s[:500]!r}")
    obj, _end = dec.raw_decode(s, start)
    if not isinstance(obj, dict):
        raise ValueError(f"Expected JSON object, got {type(obj).__name__}")
    return obj


def _normalize_score(value) -> float:
    if value is None:
        raise ValueError('Grader JSON is missing "score"')
    if isinstance(value, str):
        value = value.strip()
        try:
            return float(value)
        except ValueError as e:
            raise ValueError(f'Invalid score string: {value!r}') from e
    return float(value)


def _task_from_case(test_case) -> str:
    if isinstance(test_case, str):
        return test_case
    if not isinstance(test_case, dict):
        return str(test_case)
    return (
        test_case.get("task")
        or test_case.get("instruction")
        or test_case.get("prompt")
        or ""
    )


def grade_solution(test_case, output: str) -> dict:
    task = _task_from_case(test_case)
    eval_prompt = f"""You are an expert reviewer. Grade this solution for the task.

Task: {json.dumps(task, ensure_ascii=False)}
Solution: {json.dumps(output, ensure_ascii=False)}

Reply with ONLY a JSON object with keys:
- "strengths": array of 1-3 strings
- "weaknesses": array of 1-3 strings
- "reasoning": string
- "score": number from 1 to 10
"""
    messages = []
    add_user_message(messages, eval_prompt)
    text = chat(messages)
    return _parse_json_object(text)


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)

    if "```json" in output:
        output = output.split("```json")[1].split("```")[0].strip()
    elif "```" in output:
        output = output.split("```")[1].split("```")[0].strip()

    return json.loads(output)


def write_dataset(data, path=None):
    root = globals().get("_PROJECT", Path.cwd().resolve())
    path = Path(path) if path else root / "dataset.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    print(f"Wrote {len(data)} items to {path.resolve()}")

In [ ]:
# Optional: regenerate dataset.json (calls the API). Uncomment to run.
# dataset = generate_dataset()
# write_dataset(dataset)
# print(json.dumps(dataset, indent=2))

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result."""
    task = _task_from_case(test_case)
    if not task.strip():
        raise ValueError("test_case has no task text (expected dict with 'task' or a plain string).")
    prompt = f"""
Please solve the following task:

{task}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result with the same model."""
    output = run_prompt(test_case)
    grade = grade_solution(test_case, output)
    score = _normalize_score(grade.get("score"))
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": grade.get("reasoning", ""),
        "grade": grade,
    }

In [ ]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case."""
    results = []
    print("\n>>> Running eval — per-prompt scores print below <<<\n")
    for i, test_case in enumerate(dataset, start=1):
        result = run_test_case(test_case)
        results.append(result)
        task = _task_from_case(test_case) or str(test_case)
        preview = task if len(task) <= 100 else task[:97] + "..."
        print(f"[{i}/{len(dataset)}] score={result['score']} | {preview}")

    print("\n" + "=" * 64)
    print("PER-PROMPT SCORES (copy this block if you need a summary)")
    print("=" * 64)
    for i, r in enumerate(results, start=1):
        print(f"  Prompt {i:>2}:  score = {r['score']}")
    print("=" * 64 + "\n")
    return results

In [ ]:
import json
from pathlib import Path


def _root_for_dataset() -> Path:
    if "_PROJECT" in globals():
        return _PROJECT
    here = Path.cwd().resolve()
    for d in [here, *here.parents[:15]]:
        if (d / "code_grader.ipynb").is_file():
            return d
    for d in [here, *here.parents[:15]]:
        if (d / "dataset.json").is_file():
            return d
    return here


if "run_eval" not in globals():
    raise RuntimeError(
        "Run the cells above first, in order: 0 → 1 → 3 → 4 → 5, then run this cell again."
    )

dataset_path = _root_for_dataset() / "dataset.json"
if dataset_path.is_file():
    dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
    print(f"Loaded dataset from {dataset_path.resolve()}\n")
else:
    dataset = [
        {"task": "Write a Python function that returns True if n is even."},
        {"task": 'Write a JSON object with keys "region" and "vpc_id".'},
        {"task": "Write a regex that matches a dotted IPv4 address."},
    ]
    print(
        f"No dataset.json at {dataset_path.resolve()} — using inline sample tasks.\n"
    )

if not isinstance(dataset, list):
    raise TypeError(f"dataset must be a JSON array (list), got {type(dataset).__name__}")

results = run_eval(dataset)